# INF 201 - Deliverable_2
## Introduction
This notebook contains Python code and explanations.

## G2_2 Filmon Tsegay Kahsay and Amin Elanbri
Link to github: https://github.com/Amin12343/INF201-Deliverable-2


In [6]:
import json
import re
from pathlib import Path 

## Task 1: File ingestion and regular expressions 

In [7]:
def read_log_file(file_path: Path | str) -> tuple[str, str]: 
    """Read a text file that is encoded with either UTF-8 or ISO-8859-1

    Args: 
        file_path: Path to the file that will be read 
    
    Returns: 
        A tuple with the text content of the file and the encoding used. 
    """
    file_path = Path(file_path) # I make sure that I have a Path object 
    raw_bytes = file_path.read_bytes() #reads the whole file as bytes 

    try: # try UTF-8 first 
        text = raw_bytes.decode("utf-8") #here i turn the bytes into text 
        encoding_used = "utf-8" #remember which encoding worked 
    except UnicodeDecodeError: # the bytes are not valid UTF-8
        text = raw_bytes.decode("iso-8859-1") # here i use Latin-1 instead 
        encoding_used = "iso-8859-1" #remembers which encoding that worked 
    
    return text, encoding_used #returns both as a tuple

In [8]:
# Regular expression for one valid sales line, with one named group per field 
LOG_LINE_PATTERN = re.compile(
    r"\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] " # [2026-10-15 08:30:00]
    r"\[(?P<register_id>\w+)\] " # [POS_01]
    r"item=(?P<item>\w+) " # item=Latte
    r"price=(?P<price>\d+\.\d+) " # price=4.50
    r"(?P<currency>[A-Z]{3}) " # EUR, NOK or USD
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)$" # (SUCCESS) at the end of the line
)

In [9]:
def parse_log(text: str) -> list[dict[str, str | float]]:
    """Extract all valid sales transactions form the text of the lig file.

    Lines that do not match the valid line format (system noise) are ignored 

    Args: 
        text: The text content of a log file 

    Retuns: 
        List with one dict per valid transaction, with thw keys timestamp, 
        register_id, item, price, currency and status 
    """
    transactions = [] # her i make a list that will hold one dict per transaction 
    for line in text.splitlines(): #loops throug each line in the text 
        match = LOG_LINE_PATTERN.match(line.strip()) #try to match the pattern 
        if match is None: #the line is noise, not a sale line 
            continue 
        record = match.groupdict() #dict with the named groups as keys 
        record["price"] = float(record["price"]) #price from text to float 
        transactions.append(record) #adds the transaction to the list 

    return transactions #retuns all valid transactions 


In [10]:
# test on one of the generated log files 
test_file = Path("data/2026/library_cafe/POS_01/register_sales.log")
test_text, test_encoding = read_log_file(test_file) #reads the log 
test_transactions = parse_log(test_text) #find the valid lines 

print(f"Encoding used: {test_encoding}") #wich encoding eas used 
print(f"Number of lines in file: {len(test_text.splitlines())}") #checks all lines 
print(f"Number of valid transactions: {len(test_transactions)}") #valid lines only 
print(test_transactions[0]) #shows the first transaction 

#test that noise lines are ignored
noise_text = (
    "--- PRINTER PAPER LOW WARNING ---\n"
    "[DEBUG] BARCODE SCANNER REBOOTED\n"
    "[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)\n"
) 
assert len(parse_log(noise_text)) == 1 #only the sales line should be found 
print("Noise lines are ignored correctly")

Encoding used: utf-8
Number of lines in file: 90
Number of valid transactions: 80
{'timestamp': '2026-10-15 07:00:23', 'register_id': 'POS_01', 'item': 'Muffin', 'price': 3.84, 'currency': 'EUR', 'status': 'SUCCESS'}
Noise lines are ignored correctly


## Results and discussion 

The function read_log_file() read the test file as UTF-8. It tries UTF-8 first and only uses ISO-8859-1 if UTF-8 fails, because ISO-8859-1 never gives an error. The file was read as UTF-8 even though some files are written with ISO-8859-1. This is because the logs only contain ASCII characters, which are the same in both encodings.

The test file had 90 lines, and parse_log() found 80 valid transactions. The other 10 lines were system noise. The regular expression only matches the exact format of a sales line, so noise lines are ignored, including [DEBUG] lines that also start with a bracket.

## Task 2 Recursive utility functions 


In [11]:
def find_log_files(folder: Path | str, extension: str = ".log") -> list[Path]:
    """Find all files with the given extension in a folder and all its subfolders

    The fuction calls itself for each subfolder it finds (recursion)

    Args: 
        folder: Path to the folder to search in 
        extention: File extension to look for (default ".log")

    Returns: 
        List of Path objects for all matching files
    """
    folder = Path(folder) #makes sure that i have a Path object 
    log_files = [] #list that will hold the paths to the log files 

    for item in sorted(folder.iterdir()): #loop through everything in the folder 
        if item.is_dir(): #the item is a subfolder
            log_files = log_files + find_log_files(item, extension) #searching inside it 
        elif item.is_file() and item.name.endswith(extension): # amatching file 
            log_files.append(item) #add the file to the list 

    return log_files #return all log files that were found 

In [12]:
def read_store_config(config_path: Path | str) -> dict: 
    """Read the store configuration from JSON file 

    Args: 
        Config_path: Path to the JSON file 

    Returns: 
        The content of the files as a Python dictionary 
    """
    with open(config_path, encoding="utf-8") as f: #open the file and close it automatically 
        return json.load(f) # turn the json content into a dictonary 

In [13]:
def find_active_registers(data: dict | list) -> set[str]:
    """Find the IDs of all active registers in nested dictionaries and lists 
    
    The function calls itself for every dictionary and list it finds (recursion)

    Args: 
        data: Any combanation of nested dictionaries and lists 

    Returns: 
        A set with the IDs of all registers where "active" is true 
    """
    active_ids = set() #empty set yhat will hold the active register IDs

    if isinstance(data, dict): #the data is a dictionary 
        if "id" in data and "active" in data and data ["active"] is True:
            active_ids.add(data["id"]) #add its ID ti the set 
        for value in data.values(): #loop through all values in the dictionary 
            active_ids = active_ids | find_active_registers(value) #searching inside each value 

    elif isinstance(data, list): #the data is a list 
        for element in data: # loop trough all elements in the list 
            active_ids = active_ids | find_active_registers(element) #search inside each element 

    return active_ids #returning alll active IDs that were found 

In [14]:
log_files = find_log_files("data") #find all log files in the data folder 
print(f"Number of log files found: {len(log_files)}")

store_config = read_store_config("data/store_config.json") #her I make it read the config file
active_registers = find_active_registers(store_config) #finding the active registers 
print(f"Active registers: {active_registers}")

Number of log files found: 9
Active registers: {'POS_02', 'POS_01', 'POS_03'}


## Results and discoussion 
The function find_log_files() found all 9 log files by calling itself for every subfolder, without using globe() or os.walk(). the function find_active_registeres() found that POS_01, POS_02 and POS_03 are active, while POS_04 was left out. Because bith functions use recurision, they work no matter how deeply the folders or data are nested. 